# Stage3 / 07 — A5000 profiling of the integrated stage3 checkpoint

Clone of stage2/07 pointed at the latest stage3 run.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, sys, glob, time, json
import numpy as np, torch
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT); sys.path.insert(0, REPO_ROOT)
!pip install -q yacs thop

In [ ]:
from lib.config import cfg
from stage2.lib.models.yolopv2_detrlane import get_net_yolopv2_detrlane

YAML = sorted(glob.glob(os.path.join(REPO_ROOT, 'stage3', 'configs', 'integrated_from_*.yaml')))[-1]
cfg.defrost(); cfg.merge_from_file(YAML); cfg.freeze()
device = torch.device('cuda')

model = get_net_yolopv2_detrlane(cfg).to(device).half().eval()
ckpt = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'best.pth')
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=device)['state_dict'], strict=False)
    print('Loaded ckpt:', ckpt)

dummy = torch.randn(1, 3, 640, 640, device=device, dtype=torch.float16)
for _ in range(20):
    with torch.no_grad(): _ = model(dummy)
torch.cuda.synchronize()
lats = []
for _ in range(100):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad(): _ = model(dummy)
    torch.cuda.synchronize(); lats.append((time.perf_counter() - t0) * 1000)
lats = np.array(lats)
print(f'mean={lats.mean():.2f} ms  p50={np.median(lats):.2f} ms  FPS={1000/lats.mean():.1f}')
out = os.path.join(cfg.DRIVE.METRICS_DIR, 'a5000_profile.json')
os.makedirs(os.path.dirname(out), exist_ok=True)
with open(out, 'w') as f: json.dump({'mean_ms': float(lats.mean()),
                                     'p50_ms': float(np.median(lats)),
                                     'fps': float(1000/lats.mean())}, f, indent=2)
print('Saved ->', out)